# <font color=blue>OPTECH Use Cases: </font> 
## <font color=blue> Use Case 2</font> 
### OPTECH. Use Case 2
### Virtual network embedding based on modified genetic algorithm
#### Based on paper: Zhang, P., Yao, H., Li, M. et al. "Virtual network embedding based on modified genetic algorithm," Peer-to-Peer Netw. Appl. (2019) 12: 481.
### `CoDaS`: Erasmus Mundus master's degree in Communications, Engineering and Data Science
### `AI4CI`: Master's degree in Artificial Intelligence for Connected Industries


### <font color=#922B21>Castelldefels School of Telecommunications and Aerospace Engineering (EETAC)</font>

**<font color=teal>Cristina Cervelló-Pastor (cristina.cervello@upc.edu)</font>**

In [ ]:
# Libraries needed
import networkx as nx
import math
import random
import numpy as np
import time
import matplotlib.pyplot as plt

np.random.seed(42)
random.seed(42)

## GA Functions (Baseline)

In [ ]:
''' Calculating the fitness value of each solution in the current population '''
def cal_pop_fitness(ListOfGraphs):
    fit = []
    for G in ListOfGraphs:
        result = 0
        for link in G.edges():
            result += G.edges[link]['bw'] * G.edges[link]['hops']
        fit.append(result)
    return fit


''' Selecting the best individuals in the current generation as parents '''
def select_mating_pool(pop, num_parents):
    return pop[:num_parents].copy()


''' Single-point crossover avoiding duplicate physical nodes in the same individual (Baseline) '''
def crossover(parents, offspring_size):
    offspring = np.empty(offspring_size, dtype=int)
    # offspring_size[1] is the number of genes (num_vnodes)
    crossover_point = offspring_size[1] // 2

    # offspring_size[0] is the number of offspring to generate
    for k in range(offspring_size[0]):
        parent1_idx = k % parents.shape[0]
        parent2_idx = (k + 1) % parents.shape[0]
        
        first_half = list(parents[parent1_idx, :crossover_point])
        second_half = list(parents[parent2_idx, crossover_point:])
        
        # Combine genes without duplicating physical nodes
        child = first_half[:]
        for gene in second_half:
            if gene not in child:
                child.append(gene)
                
        # Fill remaining genes with valid unused physical nodes if duplicates were skipped
        while len(child) < offspring_size[1]:
            candidate = random.randint(1, NumNodes)
            if candidate not in child:
                child.append(candidate)
                
        offspring[k, :] = child
    return offspring


''' Baseline random mutation replacing random genes with valid unused physical node IDs '''
def mutation_baseline(offspring_crossover, num_mutations=1):
    num_genes = offspring_crossover.shape[1]
    actual_mutations = min(num_mutations, num_genes)
    
    for idx in range(offspring_crossover.shape[0]):
        genes_to_mutate = random.sample(range(num_genes), actual_mutations)
        for gene_idx in genes_to_mutate:
            available_nodes = [n for n in range(1, NumNodes + 1) if n not in offspring_crossover[idx]]
            if available_nodes:
                offspring_crossover[idx][gene_idx] = random.choice(available_nodes)
                
    return offspring_crossover


''' Evaluate if an individual mapping satisfies CPU and Bandwidth constraints '''
def feasibility(Gp, Gv, individual):
    # Reject immediately if the same physical node is used twice
    if len(set(individual)) != len(individual):
        return False, nx.Graph()

    Gvind = nx.Graph()
    
    # Track remaining bandwidth locally in case two virtual links share a physical link
    rem_bw = {e: Gp.edges[e]['bw'] for e in Gp.edges()}

    # 1. Verify if all physical nodes have enough CPU
    for i, s in enumerate(Gv.nodes()):
        p_node = int(individual[i])
        if Gp.nodes[p_node]['cpu'] >= Gv.nodes[s]['cpu']:
            Gvind.add_node(s, PNode=p_node)
        else:
            return False, Gvind
            
    # 2. Verify if there is a path with enough BW for each virtual link
    for s, t in Gv.edges():
        req_bw = Gv.edges[s, t]['bw']
        source_p = Gvind.nodes[s]['PNode']
        target_p = Gvind.nodes[t]['PNode']
        path_found = False
        
        # Check up to the 5 shortest paths
        paths_generator = nx.shortest_simple_paths(Gp, source=source_p, target=target_p)
        for _, path in zip(range(5), paths_generator):
            edges = list(zip(path, path[1:]))
            
            def get_bw(u, v):
                return rem_bw[(u, v)] if (u, v) in rem_bw else rem_bw[(v, u)]

            if all(get_bw(u, v) >= req_bw for u, v in edges):
                # Deduct bandwidth for subsequent virtual links in this VNR
                for u, v in edges:
                    if (u, v) in rem_bw:
                        rem_bw[(u, v)] -= req_bw
                    else:
                        rem_bw[(v, u)] -= req_bw
                        
                Gvind.add_edge(s, t, hops=len(path) - 1, bw=req_bw, path=path)
                path_found = True
                break  # Valid path found, move to the next virtual link
                
        if not path_found:
            return False, Gvind
            
    return True, Gvind


''' Create the initial population with 'sol_per_pop' feasible individuals '''
def InitializePopulation(sol_per_pop, num_vnodes, Gp, Gv):
    new_population = []
    ListOfGraphs = []
    attempts = 0
    max_attempts = sol_per_pop * 500  # Safety limit to prevent infinite loops
    
    while len(new_population) < sol_per_pop and attempts < max_attempts:
        attempts += 1
        individual = random.sample(range(1, NumNodes + 1), num_vnodes)
        if individual not in new_population:
            feas, G = feasibility(Gp, Gv, individual)
            if feas:
                ListOfGraphs.append(G)
                new_population.append(individual)
                
    return np.array(new_population), ListOfGraphs


''' Replace the worst individuals in the sorted population with feasible offspring '''
def UpdatePopulation(offspring_mutation, new_p, ListG, Gp, Gv):
    pop_list = new_p.tolist()
    
    for i in range(len(offspring_mutation)):
        individual = offspring_mutation[i].tolist()
        if individual not in pop_list:
            feas, G = feasibility(Gp, Gv, individual)
            if feas:
                # Remove the worst individual (assuming population is sorted ascending)
                ListG.pop()
                pop_list.pop()
                # Add the new feasible individual
                ListG.append(G)
                pop_list.append(individual)
                
    return np.array(pop_list), ListG

## Modified GA Operators (Tasks 3 & 4)

In [ ]:
## TASK 3: Partially Mapped Crossover (PMX)
def pmcrossover(parents, offspring_size):
    """
    Two-point Partially Mapped Crossover (PMX) adapted for VNE.
    
    Changes from initial draft:
    - Guarantees two distinct cut points via random.sample (non-empty cut, reproducible with random.seed).
    - Fixed second-half conflict index offset (+ crossover_range[1]).
    - Added closed-cycle fallback to select an unused substrate node from 1..NumNodes.
    """
    offspring = np.zeros(offspring_size, dtype=int)

    # offspring_size[0] is the number of offspring to generate
    for k in range(offspring_size[0]):

        # CHANGED: random.sample guarantees two distinct cut points and respects random.seed(42)
        crossover_range = sorted(random.sample(range(offspring_size[1] + 1), 2))
        
        parent1_idx = k % parents.shape[0]
        parent2_idx = (k + 1) % parents.shape[0]
        
        original_first_half = list(parents[parent1_idx, :crossover_range[0]])
        original_second_half = list(parents[parent1_idx, crossover_range[1]:])
        crossover_section = list(parents[parent2_idx, crossover_range[0]:crossover_range[1]])

        child = original_first_half + crossover_section + original_second_half
        
        if len(child) == len(set(child)):
            offspring[k, :] = child
            
        else:
            # Resolve conflicts in the first outer segment
            for i in set(crossover_section).intersection(set(original_first_half)):
                original_conflict = original_first_half.index(i)
                crossover_conflict = crossover_section.index(i) + crossover_range[0]
                
                potential_substitute = parents[parent1_idx][crossover_conflict]
                equivalence_chain = []
                resolved = False  # ADDED
                
                while potential_substitute not in equivalence_chain:
                    try:
                        equivalence_chain.append(potential_substitute)
                        next_index = child.index(potential_substitute)
                    except ValueError:
                        child[original_conflict] = potential_substitute
                        resolved = True  # ADDED
                        break
                    
                    potential_substitute = parents[parent1_idx][next_index]
                
                # ADDED: Fallback for closed cycles
                if not resolved:
                    available = [n for n in range(1, NumNodes + 1) if n not in child]
                    child[original_conflict] = random.choice(available)
            
            # Resolve conflicts in the second outer segment
            for i in set(crossover_section).intersection(set(original_second_half)):
                # CHANGED: Added '+ crossover_range[1]' offset for child index
                original_conflict = original_second_half.index(i) + crossover_range[1]
                crossover_conflict = crossover_section.index(i) + crossover_range[0]
                
                potential_substitute = parents[parent1_idx][crossover_conflict]
                equivalence_chain = []
                resolved = False  # ADDED
                
                while potential_substitute not in equivalence_chain:
                    try:
                        equivalence_chain.append(potential_substitute)
                        next_index = child.index(potential_substitute)
                    except ValueError:
                        child[original_conflict] = potential_substitute
                        resolved = True  # ADDED
                        break
                    
                    potential_substitute = parents[parent1_idx][next_index]

                # ADDED: Fallback for closed cycles
                if not resolved:
                    available = [n for n in range(1, NumNodes + 1) if n not in child]
                    child[original_conflict] = random.choice(available)

            offspring[k, :] = child
            
    return offspring

In [ ]:
## TASK 4: Resource-Aware Mutation
# CHANGED: Added Gp, Gv parameters to check CPU feasibility and residual CPU
def mutation(offspring_crossover, Gp, Gv, num_mutations=1):
    """
    Discrete, feasibility-preserving mutation operator.
    Filters unused physical nodes by CPU capacity and prioritizes higher residual CPU.
    """
    num_genes = offspring_crossover.shape[1]
    actual_mutations = min(num_mutations, num_genes)
    v_nodes = list(Gv.nodes())  # ADDED
    
    for idx in range(offspring_crossover.shape[0]):
        genes_to_mutate = random.sample(range(num_genes), actual_mutations)
        for gene_idx in genes_to_mutate:
            req_cpu = Gv.nodes[v_nodes[gene_idx]]['cpu']  # ADDED
            
            # CHANGED: Filter unused substrate nodes that satisfy CPU demand
            available_nodes = [
                n for n in range(1, NumNodes + 1)
                if n not in offspring_crossover[idx] and Gp.nodes[n]['cpu'] >= req_cpu
            ]
            if available_nodes:
                # CHANGED: Weighted selection by residual CPU for load balancing
                weights = [Gp.nodes[n]['cpu'] for n in available_nodes]
                offspring_crossover[idx][gene_idx] = random.choices(
                    available_nodes, weights=weights, k=1
                )[0]
                
    return offspring_crossover

## SCENARIO

In [ ]:
'''Generate the attributes 'location' and 'cpu' of the nodes of a network''' 
def GenerateAttribNetworkNodes(G, minCPU, maxCPU):
    distribCPU = np.random.randint(minCPU, maxCPU, len(G.nodes()))
    for n in list(G.nodes()):
        G.nodes[n]['loc'] = (0, 0)
    for n in list(G.nodes()):
        position = False
        l = [G.nodes[n]['loc'] for n in list(G.nodes())]
        while not position:
            x = np.random.randint(minX, maxX, 1)
            y = np.random.randint(minY, maxY, 1)
            if (x, y) not in l:
                G.nodes[n]['loc'] = (x, y)
                position = True
        G.nodes[n]['cpu'] = distribCPU[n]

    return G


'''Generate the attributes 'distance' and 'bw' of the links of a network''' 
def GenerateAttribNetworkLinks(G, minBW, maxBW):
    distribBW = np.random.randint(minBW, maxBW, len(G.edges()))
    for idx, link in enumerate(list(G.edges())):
        bw = int(round(distribBW[idx]))
        G.edges[(link[0], link[1])]['bw'] = bw
        G.edges[(link[1], link[0])]['bw'] = bw

    return G


'''Calculate the distance between two nodes''' 
def Distance(p1, p2):
    return round(math.sqrt((p2[0][0] - p1[0][0])**2 + (p2[1][0] - p1[1][0])**2), 2)

In [ ]:
# Location of nodes (physical and virtual)
minX = 1
maxX = 1000
minY = 1
maxY = 1000

### PHYSICAL NETWORK

In [ ]:
NumNodes = 100
Degree = 10

# Nodes CPU
minCPU = 50
maxCPU = 100

# Link BW
minBW = 50
maxBW = 100

def GeneratePhysicalNetwork():
    # Generate a random physical network with NumNodes nodes and NumNodes*Degree/2 bidirectional links
    Gp = nx.random_regular_graph(Degree, NumNodes)
    # Fill the nodes and links attributes
    GenerateAttribNetworkNodes(Gp, minCPU, maxCPU)
    GenerateAttribNetworkLinks(Gp, minBW, maxBW)

    # Relabel the number of nodes to start from 1 (instead of 0)
    Gp = nx.convert_node_labels_to_integers(Gp, first_label=1, ordering='sorted', label_attribute=None)

    return Gp

In [ ]:
# Generate the physical network
Gp = GeneratePhysicalNetwork()

# Take a copy in case it is necessary
Gpcopy = Gp.copy(as_view=False)

### VIRTUAL NETWORK

In [ ]:
minvCPU = 1
maxvCPU = 50

minvBW = 1
maxvBW = 50

minNumNodesV = 4
maxNumNodesV = 8

duration = 500  # for each request


def GenerateVirtualNetwork(NumVNodes):
    degree = (NumVNodes - 1) / 2
    # Number of nodes x degree has to be an even number
    product = math.ceil(NumVNodes * degree / 2.0) * 2
    # Thus, the degree has to be:
    degree = math.ceil(product / NumVNodes)
    Gv = nx.random_regular_graph(degree, NumVNodes)
    # Fill the nodes and links attributes
    GenerateAttribNetworkNodes(Gv, minvCPU, maxvCPU)
    GenerateAttribNetworkLinks(Gv, minvBW, maxvBW)
    # We will save the physical node where the virtual node is embedded
    for n in list(Gv.nodes()):
        Gv.nodes[n]['PNode'] = 0

    # Relabel the number of nodes to start from 1 (instead of 0)
    Gv = nx.convert_node_labels_to_integers(Gv, first_label=1, ordering='sorted', label_attribute=None)
    
    return Gv

In [ ]:
# These parameters are used in case we have several requests
VNRsucc = 0
VNRarri = 1

# Units of time of the complete simulation
TotalTime = 5000

# Parameters of a virtual network
rate = 6 / 100   # 6 requests each 100 units of time
duration = 50    # average duration for each request

# Calculate the VN request arrivals
distribRequest = np.random.poisson(rate, TotalTime)
while sum(distribRequest) == 0:
    distribRequest = np.random.poisson(rate, TotalTime)
numTotalRequest = sum(distribRequest)

# Calculate the distribution of the duration of each request
distribDuration = np.random.exponential(duration, numTotalRequest)

# Calculate the distribution of the number of nodes of each request
distribNumNodos = np.random.randint(minNumNodesV, maxNumNodesV, numTotalRequest)

In [ ]:
# Genetic algorithm function (supports both Baseline and Modified GA via `use_modified`)

def run_genetic_algorithm(Gpcopy, Gv, num_vnodes, sol_per_pop=8,
                          num_parents_mating=4, num_generations=20,
                          use_modified=True, verbose=True):
    """Run the complete Genetic Algorithm for a single Virtual Network Request (Gv)."""
    pop_size = (sol_per_pop, num_vnodes)
    
    # Initialize the population with feasible mappings
    new_population, ListOfIndividualGraphs = InitializePopulation(
        sol_per_pop, num_vnodes, Gpcopy, Gv
    )
    
    # If the physical network is too congested to form an initial population, reject the VNR
    if len(new_population) < sol_per_pop:
        if verbose:
            print("Not enough feasible individuals to form the initial population.")
        return None, None

    if verbose:
        print('Initial Population:\n', new_population, '\n')

    # Main evolutionary loop
    for generation in range(num_generations):
        if verbose:
            print(f'\n--- GENERATION: {generation} ---\n')
            
        # Measure fitness of each chromosome in the population (lower is better: bw * hops)
        fitness = np.array(cal_pop_fitness(ListOfIndividualGraphs))
        
        # Sort population and graphs from best (lowest fitness) to worst
        sorted_indices = np.argsort(fitness)
        fitness = fitness[sorted_indices]
        new_population = new_population[sorted_indices]
        ListOfIndividualGraphs = [ListOfIndividualGraphs[i] for i in sorted_indices]
        
        if verbose:
            print("Fitness:", fitness, '\n')
        
        # Select best parents in the population for mating
        parents = select_mating_pool(new_population, num_parents_mating)
        if verbose:
            print("Parents:\n", parents, '\n')
        
        # Generate next generation using crossover & mutation
        offspring_size = (pop_size[0] - len(parents), num_vnodes)
        if use_modified:
            # CHANGED: Uses PMX crossover (Task 3) and resource-aware mutation (Task 4)
            offspring_crossover = pmcrossover(parents, offspring_size=offspring_size)
            offspring_mutation = mutation(offspring_crossover, Gpcopy, Gv, num_mutations=2)
        else:
            # Baseline single-point crossover and random mutation (Task 2)
            offspring_crossover = crossover(parents, offspring_size=offspring_size)
            offspring_mutation = mutation_baseline(offspring_crossover, num_mutations=2)
        
        if verbose:
            print("Crossover:\n", offspring_crossover, '\n')
            print("Mutation:\n", offspring_mutation, '\n')
        
        # Create the new population based on the parents and mutated offspring
        new_population, ListOfIndividualGraphs = UpdatePopulation(
            offspring_mutation, new_population, ListOfIndividualGraphs, Gpcopy, Gv
        )

    # Final evaluation after finishing all generations
    fitness = np.array(cal_pop_fitness(ListOfIndividualGraphs))
    sorted_indices = np.argsort(fitness)
    
    fitness = fitness[sorted_indices]
    new_population = new_population[sorted_indices]
    ListOfIndividualGraphs = [ListOfIndividualGraphs[i] for i in sorted_indices]
    
    if verbose:
        print("\nBest solution:", new_population[0])
        print("Best fitness:", fitness[0])
    
    return new_population[0], ListOfIndividualGraphs[0]

### Single VNR Evaluation: Baseline (Task 2) vs. Modified GA (Tasks 3 & 4)

In [ ]:
sol_per_pop = 8
num_parents_mating = 4
num_generations = 20

# Number of nodes of the first virtual network based on the distribution
num_vnodes = distribNumNodos[VNRarri - 1]

# Generate the first VN request
Gv = GenerateVirtualNetwork(num_vnodes)

print("=== BASELINE GA (Single-Point Crossover + Random Mutation) ===")
t0 = time.time()
best_sol_base, best_graph_base = run_genetic_algorithm(
    Gpcopy, Gv, num_vnodes,
    sol_per_pop=sol_per_pop,
    num_parents_mating=num_parents_mating,
    num_generations=num_generations,
    use_modified=False,
    verbose=True
)
t_base = time.time() - t0
print(f"Baseline Execution Time: {t_base:.4f} s\n")

print("=== MODIFIED GA (PMX Crossover + Resource-Aware Mutation) ===")
t0 = time.time()
best_sol_mod, best_graph_mod = run_genetic_algorithm(
    Gpcopy, Gv, num_vnodes,
    sol_per_pop=sol_per_pop,
    num_parents_mating=num_parents_mating,
    num_generations=num_generations,
    use_modified=True,
    verbose=False
)
t_mod = time.time() - t0
print("Modified Best solution:", best_sol_mod)
print("Modified Best fitness:", cal_pop_fitness([best_graph_mod])[0] if best_graph_mod else None)
print(f"Modified Execution Time: {t_mod:.4f} s")

## Task 5: Dynamic Simulation Loop (Multiple VNRs)

In [ ]:
def allocate_resources(Gp, Gv, best_graph):
    """Deduct CPU and Bandwidth from Gp when a VNR is accepted."""
    for v_node in Gv.nodes():
        p_node = best_graph.nodes[v_node]['PNode']
        Gp.nodes[p_node]['cpu'] -= Gv.nodes[v_node]['cpu']

    for u, v in best_graph.edges():
        req_bw = best_graph.edges[u, v]['bw']
        path = best_graph.edges[u, v]['path']
        for a, b in zip(path, path[1:]):
            Gp.edges[a, b]['bw'] -= req_bw


def release_resources(Gp, Gv, best_graph):
    """Restore CPU and Bandwidth to Gp when a VNR expires."""
    for v_node in Gv.nodes():
        p_node = best_graph.nodes[v_node]['PNode']
        Gp.nodes[p_node]['cpu'] += Gv.nodes[v_node]['cpu']

    for u, v in best_graph.edges():
        req_bw = best_graph.edges[u, v]['bw']
        path = best_graph.edges[u, v]['path']
        for a, b in zip(path, path[1:]):
            Gp.edges[a, b]['bw'] += req_bw


def run_dynamic_simulation(Gp_initial, TotalTime, distribRequest, distribDuration, distribNumNodos,
                           sol_per_pop=8, num_parents_mating=4, num_generations=20,
                           use_modified=True, alpha=0.5, beta=0.5, log_interval=100):
    """
    Time-driven multi-VNR simulation over TotalTime:
    1. Releases CPU & BW of expired VNRs at each step t.
    2. Embeds arriving VNRs using the GA.
    3. Allocates CPU & BW on Gpcopy if accepted, schedules expiration, and logs metrics.
    """
    Gpcopy = Gp_initial.copy(as_view=False)
    active_vnrs = []  # Stores tuples of (expire_time, Gv, best_graph)

    total_init_cpu = sum(Gp_initial.nodes[n]['cpu'] for n in Gp_initial.nodes())
    total_init_bw = sum(Gp_initial.edges[e]['bw'] for e in Gp_initial.edges())

    req_idx = 0
    accepted_count = 0
    total_revenue = 0.0
    total_cost = 0.0

    # Time-series logs for plotting and Task 6 comparison
    history = {
        "time": [],
        "acceptance_ratio": [],
        "avg_revenue": [],
        "cpu_util": [],
        "bw_util": []
    }

    start_time = time.time()

    for t in range(TotalTime):
        # 1. Release resources of expired VNRs
        still_active = []
        for expire_time, Gv_act, G_emb in active_vnrs:
            if expire_time <= t:
                release_resources(Gpcopy, Gv_act, G_emb)
            else:
                still_active.append((expire_time, Gv_act, G_emb))
        active_vnrs = still_active

        # 2. Process arriving VNRs at time t
        num_arrivals = distribRequest[t]
        for _ in range(num_arrivals):
            num_vnodes = int(distribNumNodos[req_idx])
            v_duration = distribDuration[req_idx]
            req_idx += 1

            Gv = GenerateVirtualNetwork(num_vnodes)

            best_sol, best_graph = run_genetic_algorithm(
                Gpcopy, Gv, num_vnodes,
                sol_per_pop=sol_per_pop,
                num_parents_mating=num_parents_mating,
                num_generations=num_generations,
                use_modified=use_modified,
                verbose=False
            )

            if best_sol is not None:
                # Accept VNR: deduct resources and schedule expiration
                allocate_resources(Gpcopy, Gv, best_graph)
                active_vnrs.append((t + v_duration, Gv, best_graph))
                accepted_count += 1

                # Calculate Revenue and Cost (Zhang et al. Eqs. 3 & 5)
                sum_cpu = sum(Gv.nodes[n]['cpu'] for n in Gv.nodes())
                sum_bw = sum(Gv.edges[e]['bw'] for e in Gv.edges())
                sum_bw_hops = sum(
                    best_graph.edges[e]['bw'] * best_graph.edges[e]['hops']
                    for e in best_graph.edges()
                )

                total_revenue += alpha * sum_cpu + beta * sum_bw
                total_cost += alpha * sum_cpu + beta * sum_bw_hops

        # 3. Record periodic metrics
        if (t + 1) % log_interval == 0:
            rem_cpu = sum(Gpcopy.nodes[n]['cpu'] for n in Gpcopy.nodes())
            rem_bw = sum(Gpcopy.edges[e]['bw'] for e in Gpcopy.edges())
            history["time"].append(t + 1)
            history["acceptance_ratio"].append(accepted_count / req_idx if req_idx > 0 else 1.0)
            history["avg_revenue"].append(total_revenue / (t + 1))
            history["cpu_util"].append((total_init_cpu - rem_cpu) / total_init_cpu)
            history["bw_util"].append((total_init_bw - rem_bw) / total_init_bw)

    elapsed_time = time.time() - start_time
    acceptance_ratio = accepted_count / req_idx if req_idx > 0 else 0.0
    long_term_avg_revenue = total_revenue / TotalTime
    rc_ratio = total_revenue / total_cost if total_cost > 0 else 0.0
    avg_cpu_util = float(np.mean(history["cpu_util"])) if history["cpu_util"] else 0.0
    avg_bw_util = float(np.mean(history["bw_util"])) if history["bw_util"] else 0.0

    print(f"Total VNRs Arrived:       {req_idx}")
    print(f"Total VNRs Accepted:      {accepted_count}")
    print(f"Acceptance Ratio:         {acceptance_ratio:.4f}")
    print(f"Long-Term Avg Revenue:    {long_term_avg_revenue:.4f}")
    print(f"Revenue-to-Cost (R/C):    {rc_ratio:.4f}")
    print(f"Avg Node (CPU) Util:      {avg_cpu_util:.4f}")
    print(f"Avg Link (BW) Util:       {avg_bw_util:.4f}")
    print(f"Total Execution Time:     {elapsed_time:.2f} s")

    return {
        "arrived": req_idx,
        "accepted": accepted_count,
        "acceptance_ratio": acceptance_ratio,
        "avg_revenue": long_term_avg_revenue,
        "rc_ratio": rc_ratio,
        "avg_cpu_util": avg_cpu_util,
        "avg_bw_util": avg_bw_util,
        "runtime": elapsed_time,
        "history": history,
        "final_Gp": Gpcopy
    }

In [ ]:
# Run the dynamic simulation with the Modified GA
results_mga = run_dynamic_simulation(
    Gp, TotalTime, distribRequest, distribDuration, distribNumNodos,
    sol_per_pop=8, num_parents_mating=4, num_generations=20,
    use_modified=True
)

# Plot key metrics over time
hist = results_mga["history"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(hist["time"], hist["acceptance_ratio"], label="Modified GA")
axes[0].set_title("VNR Acceptance Ratio")
axes[0].set_xlabel("Time")
axes[0].set_ylabel("Acceptance Ratio")
axes[0].grid(True)

axes[1].plot(hist["time"], hist["avg_revenue"], color="green", label="Modified GA")
axes[1].set_title("Long-Term Average Revenue")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Avg Revenue")
axes[1].grid(True)

axes[2].plot(hist["time"], hist["cpu_util"], label="CPU Utilization")
axes[2].plot(hist["time"], hist["bw_util"], label="BW Utilization")
axes[2].set_title("Substrate Resource Utilization")
axes[2].set_xlabel("Time")
axes[2].set_ylabel("Utilization")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

## Task 6: Performance Comparison (Modified GA vs. Mathematical Model from Use Case 1)

Use the cell below to compare your Use Case 1 MILP/Mathematical Model results against the Modified GA on the exact same `Gp` and `Gv` instances (using `seed=42` or saved graph files).

In [ ]:
# Fill in your Use Case 1 Mathematical Model metrics (run with the exact same Gp, Gv, and seed=42)
uc1_single_cost = None      # e.g., optimal BW*hops cost from Use Case 1
uc1_single_time = None      # execution time (seconds) for single VNR

uc1_multi_metrics = {
    "acceptance_ratio": None,
    "avg_revenue": None,
    "rc_ratio": None,
    "avg_cpu_util": None,
    "avg_bw_util": None,
    "runtime": None
}

# Print comparison once Use Case 1 values are filled in
mga_single_cost = cal_pop_fitness([best_graph_mod])[0] if best_graph_mod else None
print("--- Single VNR Comparison ---")
print(f"Modified GA Cost: {mga_single_cost} | Time: {t_mod:.4f} s")
if uc1_single_cost is not None:
    gap = ((mga_single_cost - uc1_single_cost) / uc1_single_cost) * 100
    print(f"Math Model Cost:  {uc1_single_cost} | Time: {uc1_single_time:.4f} s | Optimality Gap: {gap:.2f}%")

print("\n--- Multi-VNR Comparison ---")
for metric in ["acceptance_ratio", "avg_revenue", "rc_ratio", "avg_cpu_util", "avg_bw_util", "runtime"]:
    print(f"{metric:18s} | Modified GA: {results_mga[metric]:.4f} | Math Model: {uc1_multi_metrics[metric]}")

## Lab Assignment: To-Do List

### 1. Paper Analysis
* Read **Zhang et al. (2019)** and identify how their Modified Genetic Algorithm differs from a standard GA.
* Focus specifically on **chromosome representation**, **fitness function**, **crossover**, and **mutation** operators.

### 2. Baseline Code Analysis
* Run the provided baseline code for a single Virtual Network Request (VNR).
* Analyze the evolution of **fitness**, **parents**, **crossover**, and **mutation** across generations.

### 3. Implement Partially Mapped Crossover (PMX)
* According to the paper's approach, replace the standard single-point crossover function with a **two-point Partially Mapped Crossover (PMX)** adapted for VNE.
* Preserve the relative mapping positions of non-conflicting virtual nodes while resolving duplicate physical nodes through position-based equivalence chains (with a fallback to unused substrate nodes when parents contain disjoint node subsets).

### 4. Implement Resource-Aware Mutation
* Replace the basic random mutation function with a discrete, feasibility-preserving mutation operator.
* Ensure mutated genes are replaced by **unused physical nodes** that satisfy the virtual node's **CPU requirements**, prioritizing nodes with higher residual CPU capacity for load balancing.

### 5. Dynamic Simulation Loop (Multiple VNRs)
* Implement the time-driven simulation loop using `TotalTime`, `distribRequest`, and `distribDuration`.
* **Deduct** CPU and Bandwidth from `Gpcopy` when a VNR is accepted.
* Track active VNRs and **restore** their occupied CPU and Bandwidth to `Gpcopy` once their duration expires.

### 6. Performance Comparison (GA vs. Mathematical Model)
* **Reproducibility:** Use the exact same random seed (or save/load the exact same `Gp` and `Gv` instances) used in **Use Case 1**.
* Compare both approaches across two scenarios:
  1. **Single VNR:** Optimality gap in cost/fitness and execution time.
  2. **Multiple VNRs over time:** VNR acceptance ratio, long-term average revenue, embedding cost, and node/link utilization.

### (1) Paper Analysis — Answer

Zhang et al. (2019) adapt the classical Genetic Algorithm to the NP-hard Virtual Network Embedding (VNE) problem (VNE-MGA) with the following key modifications:

- **Chromosome Representation:** Instead of binary encoding, each chromosome is an $|N_v|$-dimensional integer vector where index $i$ corresponds to virtual node $i$ and the integer value specifies its mapped substrate node (e.g., $(10, 5, 6, 8)$ maps virtual node $1 \to 10$, $2 \to 5$, etc.). All entries must be distinct so no two virtual nodes from the same VNR share a physical node.
- **Fitness Function:** Defined as the total bandwidth cost across mapped virtual links, $f(X) = \sum_{l_v \in L_v} BW(l_v) \times \text{hops}(l_v)$ (minimized; infeasible solutions are rejected or assigned $\infty$). CPU cost is omitted from the fitness objective because CPU demand is fixed/deterministic for a given VNR, whereas link bandwidth consumption depends on the variable hop count of the substrate paths.
- **Crossover (Partially Mapped Crossover — PMX):** Uses a two-point crossover between two random indices $r_1, r_2$. Because swapping a middle segment can introduce duplicate substrate nodes in the outer segments, conflicts are eliminated via position-based equivalence chains defined by the middle segment (with a fallback to unused substrate nodes if a closed cycle occurs).
- **Mutation:** Rather than purely random gene replacement, Zhang et al. (2019) propose an *improved mutation* that remaps selected virtual nodes to unused substrate nodes that satisfy CPU requirements and are **closer to the already-mapped substrate nodes** to reduce hop count and bandwidth waste. (In this lab's Task 4 adaptation, mutation is further tailored for load balancing by prioritizing unused CPU-feasible substrate nodes with higher residual CPU capacity.)
- **Additional Paper Contributions:** Zhang et al. also introduce a **hybrid population initialization** strategy (generating one high-quality solution and mutating it to form the rest of the population) and a post-mutation **improvement operation** that remaps a node to a 1-hop neighbor of the mapped set to accelerate convergence.

### (2) Baseline Code Analysis — Answer

Running the baseline GA (`crossover` + `mutation_baseline`) on the first single VNR with `seed=42`:

- **Fitness:** Fitness improves very little (or stagnates completely—e.g., starting at `294` and remaining at `294` after 20 generations). Additionally, because `UpdatePopulation` pops the worst individual and inserts *any* feasible offspring without verifying that the offspring has better fitness than the removed individual, the worst-case fitness in the population can temporarily worsen across generations.
- **Parents:** Because `select_mating_pool` uses strict elitism (always selecting the top 4 individuals), the mating pool quickly becomes repetitive when few better offspring are discovered, reducing genetic diversity and slowing exploration.
- **Crossover:** The baseline uses a fixed midpoint single-point crossover (`offspring_size[1] // 2`) with random node fill when duplicates occur. This disrupts valid sub-mappings and does not preserve relative node relationships between parents.
- **Mutation:** The baseline randomly replaces genes with any unused physical node without checking CPU feasibility or residual capacity. As a result, many mutated offspring fail `feasibility()` or increase path hop counts, leading to wasted evaluations and poor convergence.